Merscope output documentation
https://vizgen.com/wp-content/uploads/2023/06/91600001_MERSCOPE-Instrument-User-Guide_Rev-G.pdf

The cell_boundaries.parquet file
contains the boundaries of cells in microns or pixels, formatted as a data table using GeoPandas

**This script process vizgen .parquet file that has cell segmentation to generate cell segmentation tiff img**

input: .parquet file Geometry column using MULTIPOLYGON for cell segmentation

output: cell segmentation outline in tiff file

In [1]:
import pandas as pd
import geopandas as gpd
import numpy
import cv2
import skimage.io

In [8]:
parquet_file="mosaic_space.parquet" # pixel coordinate
Z = 3 # Select ZIndex of the middle z plan

outlineColor = 200
output = 'z3_outline_output.tiff'

In [3]:
gdf_mosaic = gpd.read_parquet(parquet_file)
gdf_mosaic

,ID,EntityID,ZIndex,Geometry,Type,ZLevel,Name,ParentID,ParentType
0,0,1578230100020100001,0,"MULTIPOLYGON (((24619.12 919.884, 24618.693 92...",cell,1.5,None,None,None
1,1,1578230100020100003,0,"MULTIPOLYGON (((25052.061 857.342, 25059.336 8...",cell,1.5,None,None,None
4,2,1578230100020100007,0,"MULTIPOLYGON (((24438.027 862.793, 24451.707 8...",cell,1.5,None,None,None
5,3,1578230100020100008,0,"MULTIPOLYGON (((24795.622 845.575, 24805.649 8...",cell,1.5,None,None,None
6,4,1578230100020100009,0,"MULTIPOLYGON (((24375.573 877.53, 24382.811 89...",cell,1.5,None,None,None
...,...,...,...,...,...,...,...,...,...
437499,387412,1578230103360100035,6,"MULTIPOLYGON (((65706.369 68570.631, 65704.103...",cell,10.5,None,None,None
437500,387413,1578230103360100029,6,"MULTIPOLYGON (((65834 68608.564, 65836.13 6862...",cell,10.5,None,None,None
437501,387414,1578230103360100036,6,"MULTIPOLYGON (((65300.655 68589.529, 65305.942...",cell,10.5,None,None,None
437502,387415,1578230103361100003,0,"MULTIPOLYGON (((66017.624 68454.577, 66026.679...",cell,1.5,None,None,None


In [4]:
minx, miny, maxx, maxy = gdf_mosaic.total_bounds
scale = 10
minx, miny, maxx, maxy

(47.68746730934208, 17.0, 69728.0, 69750.93880737045)

In [6]:
z_levels = gdf_mosaic["ZIndex"].value_counts().sort_index()
z_levels

ZIndex
0    52485
1    53289
2    56503
3    58648
4    58316
5    55391
6    52771
Name: count, dtype: int64

# single z level outline of masks to tiff file 

In [9]:
df = gdf_mosaic[gdf_mosaic['ZIndex'] == Z]
df

,ID,EntityID,ZIndex,Geometry,Type,ZLevel,Name,ParentID,ParentType
125,113,1578230100020100051,3,"MULTIPOLYGON (((24598.774 900.487, 24600.4 921...",cell,6.0,None,None,None
126,114,1578230100020100003,3,"MULTIPOLYGON (((25084.248 905.92, 25091.129 89...",cell,6.0,None,None,None
127,115,1578230100020100070,3,"MULTIPOLYGON (((24263.624 859.577, 24272.342 8...",cell,6.0,None,None,None
128,116,1578230100020100005,3,"MULTIPOLYGON (((25237 858.564, 25244.057 884.0...",cell,6.0,None,None,None
129,117,1578230100020100053,3,"MULTIPOLYGON (((24437.417 871.402, 24441.473 8...",cell,6.0,None,None,None
...,...,...,...,...,...,...,...,...,...
437469,387384,1578230103360100017,3,"MULTIPOLYGON (((65089.885 68610.649, 65092.415...",cell,6.0,None,None,None
437470,387385,1578230103360100027,3,"MULTIPOLYGON (((65023.121 68575.121, 65024.061...",cell,6.0,None,None,None
437471,387386,1578230103360100031,3,"MULTIPOLYGON (((65621.389 68612.611, 65619 686...",cell,6.0,None,None,None
437472,387387,1578230103360100029,3,"MULTIPOLYGON (((65833 68608.564, 65835.13 6862...",cell,6.0,None,None,None


In [10]:
height = 69762 #info obtained from measuring the mosic image
width = 69741

# Initialize a blank image (mask)
mask = numpy.zeros((height, width), dtype=numpy.uint8)
mask

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=uint8)

In [11]:
# Draw the polygons as the mask outline
for idx, row in df.iterrows():
    geometry = row['Geometry']  # Get the geometry (Polygon/MultiPolygon)
    fill_value = int(row['EntityID'])  # Replace 'value_column' with the actual column name

    for polygon in geometry.geoms:
        exterior_coords = numpy.array([[int(x), int(y)] for x, y in polygon.exterior.coords], dtype=numpy.int32)
        cv2.polylines(mask, [exterior_coords], isClosed=True, color=outlineColor)
mask

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=uint8)

In [12]:
# save outline to tiff file

skimage.io.imsave(output, mask)

/var/folders/kl/2d3p5js5705bjqlv6qpr9_c00000gn/T/ipykernel_22355/1123878571.py:3: UserWarning: z3_outline_output.tiff is a low contrast image
  skimage.io.imsave(output, mask)
